## Data Preprocessing

In [1]:
# Importing the basic libraries for data manipulation and preprocessing

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split

In [2]:
# Load the dataset
df = pd.read_csv('../../data/raw/ai4i2020.csv')

# Step 1: Check for missing values
print("Missing values per column:")
print(df.isna().sum())
print("\n" + "="*60 + "\n")

Missing values per column:
UID                    0
Product ID             0
Type                   0
Air temperature        0
Process temperature    0
Rotational speed       0
Torque                 0
Tool wear              0
Machine failure        0
TWF                    0
HDF                    0
PWF                    0
OSF                    0
RNF                    0
dtype: int64




Since there is no missing values, we are good to move to next step

In [9]:
# Step 2: Extract product quality variant from Product ID
# Product ID format: e.g., "L50001" -> quality = "L", serial = "50001"
df["product_quality"] = df["Product ID"].str[0]  # Extract first letter
print("Unique product quality variants:")
print(df["product_quality"].unique())
print("Product quality distribution:")
print(df["product_quality"].value_counts())


Unique product quality variants:
<StringArray>
['M', 'L', 'H']
Length: 3, dtype: str
Product quality distribution:
product_quality
L    6000
M    2997
H    1003
Name: count, dtype: int64


Why: Quality variant matters because:

Different quality products accumulate tool wear differently (L adds 2 min, M adds 3 min, H adds 5 min).
This influences failure modes like overstrain.
The model needs to know product quality to predict correctly.

In [11]:
# Step 3: Check target variable balance BEFORE preprocessing
print("Target variable balance (before split):")
print(df["Machine failure"].value_counts())
print(f"Failure rate: {df['Machine failure'].mean():.2%}")

Target variable balance (before split):
Machine failure
0    9661
1     339
Name: count, dtype: int64
Failure rate: 3.39%


In [34]:
# Since, there is a significant imbalance in the target variable, we will need to address this issue during the data preprocessing steps. 
# Only 3.39% of the samples represent machine failures, which could lead to biased model performance if not handled properly. 
# A naive model that predicts "no failure" for everything would be 96% accurate but useless. 
# This is why recall is more important than accuracy: missing a failure is costly and unsafe.
# We might need to use techniques like: class_weight='balanced' in the model, undersampling the majority class, oversampling the minority class.

In [ ]:
# Step 4: Preparing features and target

X = df.drop(columns=["UID", "Product ID", "Machine failure"])
y = df["Machine failure"]

print(f"Features shape before encoding: {X.shape}")
print(f"Features columns: {X.columns.tolist()}")
print("\n" + "="*60 + "\n")

Features shape before encoding: (10000, 12)
Features columns: ['Type', 'Air temperature', 'Process temperature', 'Rotational speed', 'Torque', 'Tool wear', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF', 'product_quality']




In [37]:
# Step 5: Encode categorical features (product_quality)

X_encoded = pd.get_dummies(X, columns=["product_quality"], drop_first=True, dtype=int)
print(f"Features shape after one-hot encoding: {X_encoded.shape}")
print(f"New feature columns: {X_encoded.columns.tolist()}")

Features shape after one-hot encoding: (10000, 13)
New feature columns: ['Type', 'Air temperature', 'Process temperature', 'Rotational speed', 'Torque', 'Tool wear', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF', 'product_quality_L', 'product_quality_M']


There are 12 input features:

Sensor readings: Air temperature, Process temperature, Rotational speed, Torque, Tool wear

Failure mode indicators: TWF, HDF, PWF, OSF, RNF (binary flags showing if each mode occurred)

Categorical: Type (product type), product_quality

One-hot encoding converts product_quality (L/M/H) into numeric columns:

product_quality_L: 1 if L, else 0
    
product_quality_M: 1 if M, else 0

(H is implicit: 0 for both means H)


# Step 6: Scale numeric features
Raw values have different units:

Temperature: ~300 K,
Speed: ~2000 rpm,
Torque: ~40 Nm,
Tool wear: ~100 min
After standardization:

Ex: Air temperature: -0.952389  (was ~298 K, now normalized),
    Torque:  0.282200           (was ~41 Nm, now normalized)

We are using standard scalar library from sklearn which scales each features within ~N(0, 1).

In [44]:

numeric_cols = X_encoded.select_dtypes(include="number").columns.tolist()
print(f"Numeric feature columns: {numeric_cols}\n")

scaler = StandardScaler()
X_scaled = X_encoded.copy()
X_scaled[numeric_cols] = scaler.fit_transform(X_encoded[numeric_cols])

print("Scaled data (first 5 rows):")
print(X_scaled.head())
print("\n" + "="*60 + "\n")


Numeric feature columns: ['Air temperature', 'Process temperature', 'Rotational speed', 'Torque', 'Tool wear', 'TWF', 'HDF', 'PWF', 'OSF', 'RNF', 'product_quality_L', 'product_quality_M']

Scaled data (first 5 rows):
  Type  Air temperature  Process temperature  Rotational speed    Torque  \
0    M        -0.952389            -0.947360          0.068185  0.282200   
1    L        -0.902393            -0.879959         -0.729472  0.633308   
2    L        -0.952389            -1.014761         -0.227450  0.944290   
3    L        -0.902393            -0.947360         -0.590021 -0.048845   
4    L        -0.902393            -0.879959         -0.729472  0.001313   

   Tool wear      TWF      HDF       PWF       OSF      RNF  \
0  -1.695984 -0.06798 -0.10786 -0.097934 -0.099484 -0.04363   
1  -1.648852 -0.06798 -0.10786 -0.097934 -0.099484 -0.04363   
2  -1.617430 -0.06798 -0.10786 -0.097934 -0.099484 -0.04363   
3  -1.586009 -0.06798 -0.10786 -0.097934 -0.099484 -0.04363   
4  -1.55458

# Step 7: Stratified train/test split

Key detail: stratified split

stratify=y ensures the failure proportion stays ~3.39% in both train and test.

Without stratification, the test set might have 1% failures (not representative) or 10% (biased).

This is critical for imbalanced data to ensure fair evaluation.

In [46]:
# Step 7: Split data into train/test with stratification
# Stratification keeps failure/no-failure proportions consistent in both sets

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")
print(f"\nTraining set failure rate: {y_train.mean():.2%}")
print(f"Test set failure rate: {y_test.mean():.2%}")

Training set size: 8000 samples
Test set size: 2000 samples

Training set failure rate: 3.39%
Test set failure rate: 3.40%


In [47]:
# Step 8: Summary statistics after scaling
print("Scaled features summary (first 5 rows):")
print(X_train.head())
print(f"\nFeatures used for modeling: {X_train.columns.tolist()}")

Scaled features summary (first 5 rows):
     Type  Air temperature  Process temperature  Rotational speed    Torque  \
4058    M         0.997456             0.602860         -0.461727  0.723593   
1221    M        -1.502346            -1.149563         -0.779674  0.643340   
6895    M         0.497496             1.074667         -1.013950  0.563086   
9863    L        -0.552421            -0.138549         -0.712737  1.636474   
8711    L        -1.452350            -1.014761          1.083385 -1.132265   

      Tool wear      TWF      HDF       PWF       OSF        RNF  \
4058  -0.847607 -0.06798 -0.10786 -0.097934 -0.099484  -0.043630   
1221   0.377826 -0.06798 -0.10786 -0.097934 -0.099484  22.919769   
6895   0.456380 -0.06798 -0.10786 -0.097934 -0.099484  -0.043630   
9863  -0.376287 -0.06798 -0.10786 -0.097934 -0.099484  -0.043630   
8711  -0.910450 -0.06798 -0.10786 -0.097934 -0.099484  -0.043630   

      product_quality_L  product_quality_M  
4058          -1.224745        